<a href="https://colab.research.google.com/github/dia-rodriguez/optics-transfer-matrix-sim/blob/main/Transfer_Matrix_Method_optics_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Core functions (like lego bricks for fast calculations)

In [ ]:
import numpy as np
np.set_printoptions(suppress=True, precision=16)

def R_term_spherical(n,n_prime,R):
  """ this is refraction matrix for spherical interface thick lens (the one from previous activities)
      note: R is different from R_term funcs (these are matrices and R var
      is just the radius)

  """
  return np.array([ [1, 0], [(n-n_prime) / (n_prime*R), n / n_prime] ])

def T_term(t):
  """ this function can acommodate thickness or distance (d and t variables)
      this is translation matrix
  """
  return np.array([[1, t],[0, 1]])

def R_term_plane(n,n_prime):
  """ this is refraction matrix for plane interface"""
  return np.array([ [1, 0], [0, n / n_prime] ])

def thin_lens_term(n,n_prime,R1,R2):
  one_over_f = (n_prime - n) / n * (1/R1 - 1/R2)
  return np.array([[1, 0], [-(one_over_f), 1]])

def thin_lens_f_term(f):
    """ thin lens matrix when focal length f is given directly """
    return np.array([[1, 0], [-1/f, 1]])

def spherical_mirror_term(R):
  return np.array([[1, 0], [2/R, 1]])

def M_sys(*M):
  """ this automatically multiply matrices automatically for M_sys by using args
      note: msys multiplies matrices from right to left
      M_sys = MN * M3 * M2 * M1
      M_sys = np.linalg.multi_dot([M_last, M_middle, M_first]) numpy does same

  """
  return np.linalg.multi_dot(M)

#just in case
def generate_ray_bundle(y0, theta_min, theta_max, num_rays=50):
    """Generates a 2xN matrix of initial rays [y0, theta0]."""
    theta_0 = np.linspace(theta_min, theta_max, num_rays)
    y_0 = np.full_like(theta_0, y0)
    return np.vstack([y_0, theta_0])  # Matrix of shape (2, num_rays)

def trace_rays(M_sys, V_in):
    """Multiplies M_sys by the ray matrix V_in to get output rays V_out."""
    return M_sys @ V_in  # Returns matrix of shape (2, num_rays)

Table 1

In [ ]:
def run_flat_interface():
  """ some updates to code:
    - before it was just: n_prime = 1 n = 1.6 now change them respectively to
      n_in and n_out to keep track of optics order and keep n,n_prime
      in our core functions not inside simulation blocks

      - previous version:
      n_prime = 1 #this is just air medium
      n = 1.6
      flat_interface_R = R_term_plane(n,n_prime)
      because n_prime or n_in came first
  """
  d = 0.08
  n_in = 1 #this is just air medium
  n_out = 1.6
  R = np.inf
  flat_interface_T = T_term(d)
  flat_interface_R = R_term_plane(n_in,n_out)
  flat_interface_M = M_sys(flat_interface_R, flat_interface_T)
  return flat_interface_T, flat_interface_R, flat_interface_M

#print for now
flat_matrices = run_flat_interface()
print("Translation Matrix (T):")
print(flat_matrices[0])
print("Refraction Matrix (R):")
print(flat_matrices[1])
print("System Matrix (M):")
print(flat_matrices[2])

Translation Matrix (T):
[[1.   0.08]
 [0.   1.  ]]
Refraction Matrix (R):
[[1.    0.   ]
 [0.    0.625]]
System Matrix (M):
[[1.    0.08 ]
 [0.    0.625]]


In [ ]:
def run_convex_interface():
  d = 0.08
  n_in = 1
  n_out = 1.6
  R = 0.05
  convex_interface_T = T_term(d)
  convex_interface_R = R_term_spherical(n_in,n_out,R)
  convex_interface_M = M_sys(convex_interface_R, convex_interface_T)
  return convex_interface_T, convex_interface_R, convex_interface_M

convex_matrices = run_convex_interface()
print("Translation Matrix (T):")
print(convex_matrices[0])
print("Refraction Matrix (R):")
print(convex_matrices[1])
print("System Matrix (M):")
print(convex_matrices[2])

Translation Matrix (T):
[[1.   0.08]
 [0.   1.  ]]
Refraction Matrix (R):
[[ 1.     0.   ]
 [-7.5    0.625]]
System Matrix (M):
[[ 1.     0.08 ]
 [-7.5    0.025]]


In [ ]:
def run_concave_interface():
  """ same but sign convention is the only difference"""
  d = 0.08
  n_in = 1
  n_out = 1.6
  R = -0.05
  concave_interface_T = T_term(d)
  concave_interface_R = R_term_spherical(n_in,n_out,R)
  concave_interface_M = M_sys(concave_interface_R, concave_interface_T)
  return concave_interface_T, concave_interface_R, concave_interface_M

concave_matrices = run_concave_interface()
print("Translation Matrix (T):")
print(concave_matrices[0])
print("Refraction Matrix (R):")
print(concave_matrices[1])
print("System Matrix (M):")
print(concave_matrices[2])

Translation Matrix (T):
[[1.   0.08]
 [0.   1.  ]]
Refraction Matrix (R):
[[1.    0.   ]
 [7.5   0.625]]
System Matrix (M):
[[1.    0.08 ]
 [7.5   1.225]]


Table 2

In [ ]:
def run_biconvex_lens():
  """ order optics to keep track:
      biconvex_T > biconvex_R1 > biconvex_T_glass > biconvex_R2

      - since d is given so i thought T_term from outside comes first
      then refraction, then thickness since it has t var then refraction again

      - update: flipping values for R2 matrix (n_out,n_in,R2)
      bc we are refract out of the lens: glass(n_out)>air(n_in) we are going out

  """
  d = 0.08
  t = 0.02
  n_in = 1 #n_air
  n_out = 1.6 #n_g glass
  R1 = 0.05
  R2 = -0.03
  biconvex_T = T_term(d)
  biconvex_R1 = R_term_spherical(n_in,n_out,R1)
  biconvex_T_glass = T_term(t)
  biconvex_R2 = R_term_spherical(n_out,n_in,R2)
  biconvex_M = M_sys(biconvex_R2, biconvex_T_glass, biconvex_R1, biconvex_T)
  return biconvex_T, biconvex_R1, biconvex_T_glass, biconvex_R2, biconvex_M

  #at this point im gonna print Msys for now, theres alot of matrices to print


biconvex_matrices = run_biconvex_lens()

for i in range(len(biconvex_matrices)-1):
  print(f"Element Matrix {i+1}:")
  print(biconvex_matrices[i])
  print("\n")

print("System Matrix (M):")
print(biconvex_matrices[-1])

Element Matrix 1:
[[1.   0.08]
 [0.   1.  ]]


Element Matrix 2:
[[ 1.     0.   ]
 [-7.5    0.625]]


Element Matrix 3:
[[1.   0.02]
 [0.   1.  ]]


Element Matrix 4:
[[  1.                  0.               ]
 [-20.000000000000004   1.6              ]]


System Matrix (M):
[[  0.85                 0.0805            ]
 [-29.000000000000004   -1.5700000000000003]]


In [ ]:
def run_biconcave_lens():
  d = 0.08
  t = 0.02
  n_in = 1 #n_air
  n_out = 1.6 #n_g glass
  R1 = -0.05
  R2 = 0.03
  biconcave_T = T_term(d)
  biconcave_R1 = R_term_spherical(n_in,n_out,R1)
  biconcave_T_glass = T_term(t)
  biconcave_R2 = R_term_spherical(n_out,n_in,R2)
  biconcave_M = M_sys(biconcave_R2, biconcave_T_glass, biconcave_R1, biconcave_T)
  return biconcave_T, biconcave_R1, biconcave_T_glass, biconcave_R2, biconcave_M

biconcave_matrices = run_biconcave_lens()

for i in range(len(biconcave_matrices)-1):
  print(f"Element Matrix {i+1}:")
  print(biconcave_matrices[i])
  print("\n")

print("System Matrix (M):")
print(biconcave_matrices[-1])

Element Matrix 1:
[[1.   0.08]
 [0.   1.  ]]


Element Matrix 2:
[[1.    0.   ]
 [7.5   0.625]]


Element Matrix 3:
[[1.   0.02]
 [0.   1.  ]]


Element Matrix 4:
[[ 1.                 0.               ]
 [20.000000000000004  1.6              ]]


System Matrix (M):
[[ 1.15               0.1045           ]
 [35.00000000000001   4.050000000000001]]


In [ ]:
def run_compound_lens():
  """ this is basically a doublet
      path: T1>R1>T2>R2
  """
  d1 = 0.08
  n1 = 1.6
  R1 = 0.1
  n2 = 2
  R2 = -0.2
  d2 = 0.02
  n3 = 1

  compound_T1 = T_term(d1)
  compound_R1 = R_term_spherical(n1, n2, R1)
  compound_T2 = T_term(d2)
  compound_R2 = R_term_spherical(n2, n3, R2)
  compound_M = M_sys(compound_R2, compound_T2, compound_R1, compound_T1)
  return compound_T1, compound_R1, compound_T2, compound_R2, compound_M

compound_matrices = run_compound_lens()

for i in range(len(compound_matrices)-1):
  print(f"Element Matrix {i+1}:")
  print(compound_matrices[i])
  print("\n")

print("System Matrix (M):")
print(compound_matrices[-1])

Element Matrix 1:
[[1.   0.08]
 [0.   1.  ]]


Element Matrix 2:
[[ 1.                  0.                ]
 [-1.9999999999999996  0.8               ]]


Element Matrix 3:
[[1.   0.02]
 [0.   1.  ]]


Element Matrix 4:
[[ 1.  0.]
 [-5.  2.]]


System Matrix (M):
[[ 0.96                0.0928            ]
 [-8.799999999999999   0.8160000000000003]]


In [ ]:
def run_condensor_lens():
    d = 0.08
    f = 0.04
    condensor_T = T_term(d)
    condensor_L = thin_lens_f_term(f)
    condensor_M = M_sys(condensor_L, condensor_T)
    return condensor_T, condensor_L, condensor_M

condensor_matrices = run_condensor_lens()

for i in range(len(condensor_matrices)-1):
  print(f"Element Matrix {i+1}:")
  print(condensor_matrices[i])
  print("\n")

print("System Matrix (M):")
print(condensor_matrices[-1])

Element Matrix 1:
[[1.   0.08]
 [0.   1.  ]]


Element Matrix 2:
[[  1.   0.]
 [-25.   1.]]


System Matrix (M):
[[  1.     0.08]
 [-25.    -1.  ]]


In [ ]:
def run_immersion_lens():
    """
    optical path: T_cover > R_flat > T_flat > R1 > T_lens > R2 > T_air

    """
    t = 0.00005
    n = 1.6

    d_flat = 0.08
    n_flat = 1.33

    R1 = 0.02
    n_g = 1.52
    d_lens = 0.015
    R2 = -0.05

    n_air = 1
    d_air = 0.005

    imm_T_cover = T_term(t)
    imm_R_flat = R_term_plane(n, n_flat)
    imm_T_flat = T_term(d_flat)
    imm_R1 = R_term_spherical(n_flat, n_g, R1)
    imm_T_lens = T_term(d_lens)
    imm_R2 = R_term_spherical(n_g, n_air, R2)
    imm_T_air = T_term(d_air)
    imm_M = M_sys(imm_T_air, imm_R2, imm_T_lens, imm_R1, imm_T_flat, imm_R_flat, imm_T_cover)
    return imm_T_cover, imm_R_flat, imm_T_flat, imm_R1, imm_T_lens, imm_R2, imm_T_air, imm_M

# Run it
immersion_matrices = run_immersion_lens()

for i in range(len(immersion_matrices)-1):
  print(f"Element Matrix {i+1}:")
  print(immersion_matrices[i])
  print("\n")

print("Immersion Lens System Matrix (M):")
print(immersion_matrices[-1])

Element Matrix 1:
[[1.      0.00005]
 [0.      1.     ]]


Element Matrix 2:
[[1.                 0.                ]
 [0.                 1.2030075187969924]]


Element Matrix 3:
[[1.   0.08]
 [0.   1.  ]]


Element Matrix 4:
[[ 1.                 0.               ]
 [-6.249999999999998  0.875            ]]


Element Matrix 5:
[[1.    0.015]
 [0.    1.   ]]


Element Matrix 6:
[[  1.     0.  ]
 [-10.4    1.52]]


Element Matrix 7:
[[1.    0.005]
 [0.    1.   ]]


Immersion Lens System Matrix (M):
[[  0.811625             0.1011202804981203]
 [-18.924999999999997   -0.386510159774436 ]]


Table 3

In [ ]:
def run_eyes():
    """
      note: i noticed that i can just look at the elements on
      the web simulation for each of the path so theres less
      confusion on my part

      path: T > R_spherical > T > R_spherical > T > R_thin_lens > T > R_spherical > T > R_spherical
    """
    n_air = 1.0
#rn putting comments on each element so im not confussed
    # Element 1
    d1 = 0.09

    # Element 2
    n1 = 2.0
    R1 = 0.3

    # Element 3
    d2 = 0.002

    # Element 4
    n2 = 1.0
    R2 = 0.2

    # Element 5
    d3 = 0.01

    # Element 6
    f = 0.025

    # Element 7
    d4 = 0.025

    # Element 8
    n3 = 10.0
    R3 = -0.05

    # Element 9
    d5 = 0.0001

    # Element 10
    n4 = 1.0
    R4 = -0.05

    eyes_T1 = T_term(d1)
    eyes_R1 = R_term_spherical(n_air, n1, R1)

    eyes_T2 = T_term(d2)
    eyes_R2 = R_term_spherical(n1, n2, R2)

    eyes_T3 = T_term(d3)
    eyes_L  = thin_lens_f_term(f)

    eyes_T4 = T_term(d4)
    eyes_R3 = R_term_spherical(n2, n3, R3)

    eyes_T5 = T_term(d5)
    eyes_R4 = R_term_spherical(n3, n4, R4)

    eyes_M = M_sys(eyes_R4, eyes_T5, eyes_R3, eyes_T4, eyes_L, eyes_T3, eyes_R2, eyes_T2, eyes_R1, eyes_T1)
    return eyes_T1, eyes_R1, eyes_T2, eyes_R2, eyes_T3, eyes_L, eyes_T4, eyes_R3, eyes_T5, eyes_R4, eyes_M

eye_matrices = run_eyes()

for i in range(len(eye_matrices)-1):
  print(f"Element Matrix {i+1}:")
  print(eye_matrices[i])
  print("\n")

print("System Matrix (M):")
print(eye_matrices[-1])

Element Matrix 1:
[[1.   0.09]
 [0.   1.  ]]


Element Matrix 2:
[[ 1.                  0.                ]
 [-1.6666666666666667  0.5               ]]


Element Matrix 3:
[[1.    0.002]
 [0.    1.   ]]


Element Matrix 4:
[[1. 0.]
 [5. 2.]]


Element Matrix 5:
[[1.   0.01]
 [0.   1.  ]]


Element Matrix 6:
[[  1.   0.]
 [-40.   1.]]


Element Matrix 7:
[[1.    0.025]
 [0.    1.   ]]


Element Matrix 8:
[[ 1.   0. ]
 [18.   0.1]]


Element Matrix 9:
[[1.     0.0001]
 [0.     1.    ]]


Element Matrix 10:
[[   1.    0.]
 [-180.   10.]]


System Matrix (M):
[[  0.0409354833333332   0.0288600485      ]
 [-38.820053666666674   -2.939958730000001 ]]


In [ ]:
def run_microscope():
    """
    path: T > R_thin_lens > T > R_thin_lens > T > R_thin_lens > T > R_spherical > T > R_spherical
    """
    n_air = 1.0

    # Element 1
    d1 = 0.017

    # Element 2
    f1 = 0.0145

    # Element 3
    d2 = 0.1185

    # Element 4
    f2 = 0.02

    # Element 5
    d3 = 0.01

    # Element 6
    f = 0.025

    # Element 7
    d4 = 0.025

    # Element 8
    n1 = 10.0
    R1 = -0.05

    # Element 9
    d5 = 0.0001

    # Element 10
    n2 = 1.0
    R2 = -0.05

    microscope_T1 = T_term(d1)
    microscope_L1 = thin_lens_f_term(f1)

    microscope_T2 = T_term(d2)
    microscope_L2 = thin_lens_f_term(f2)

    microscope_T3 = T_term(d3)
    microscope_L3 = thin_lens_f_term(f)

    microscope_T4 = T_term(d4)
    microscope_R1 = R_term_spherical(n_air, n1, R1)

    microscope_T5 = T_term(d5)
    microscope_R2 = R_term_spherical(n1, n2, R2)

    microscope_M = M_sys(microscope_R2, microscope_T5, microscope_R1, microscope_T4, microscope_L3, microscope_T3, microscope_L2, microscope_T2, microscope_L1, microscope_T1)

    return microscope_T1, microscope_L1, microscope_T2, microscope_L2, microscope_T3, microscope_L3, microscope_T4, microscope_R1, microscope_T5, microscope_R2, microscope_M

microscope_matrices = run_microscope()

for i in range(len(microscope_matrices)-1):
  print(f"Element Matrix {i+1}:")
  print(microscope_matrices[i])
  print("\n")

print("System Matrix (M):")
print(microscope_matrices[-1])

Element Matrix 1:
[[1.    0.017]
 [0.    1.   ]]


Element Matrix 2:
[[  1.                0.             ]
 [-68.9655172413793   1.             ]]


Element Matrix 3:
[[1.     0.1185]
 [0.     1.    ]]


Element Matrix 4:
[[  1.   0.]
 [-50.   1.]]


Element Matrix 5:
[[1.   0.01]
 [0.   1.  ]]


Element Matrix 6:
[[  1.   0.]
 [-40.   1.]]


Element Matrix 7:
[[1.    0.025]
 [0.    1.   ]]


Element Matrix 8:
[[ 1.   0. ]
 [18.   0.1]]


Element Matrix 9:
[[1.     0.0001]
 [0.     1.    ]]


Element Matrix 10:
[[   1.    0.]
 [-180.   10.]]


System Matrix (M):
[[  7.259020689655172   -0.0000202232758621]
 [457.51420689655174     0.1364850172413784]]


In [ ]:
def run_keplerian():
    """
    I noticed that keplerian and gallilean telescopes have similar structure as
    microscope but just different values so i just copy cause brain tired
    """
    n_air = 1.0

    # Element 1
    d1 = 0.02

    # Element 2
    f1 = 0.08

    # Element 3
    d2 = 0.1

    # Element 4
    f2 = 0.02

    # Element 5
    d3 = 0.01

    # Element 6
    f = 0.025

    # Element 7
    d4 = 0.025

    # Element 8
    n1 = 10.0
    R1 = -0.05

    # Element 9
    d5 = 0.0001

    # Element 10
    n2 = 1.0
    R2 = -0.05

    keplerian_T1 = T_term(d1)
    keplerian_L1 = thin_lens_f_term(f1)

    keplerian_T2 = T_term(d2)
    keplerian_L2 = thin_lens_f_term(f2)

    keplerian_T3 = T_term(d3)
    keplerian_L3 = thin_lens_f_term(f)

    keplerian_T4 = T_term(d4)
    keplerian_R1 = R_term_spherical(n_air, n1, R1)

    keplerian_T5 = T_term(d5)
    keplerian_R2 = R_term_spherical(n1, n2, R2)

    keplerian_M = M_sys(keplerian_R2, keplerian_T5, keplerian_R1, keplerian_T4, keplerian_L3, keplerian_T3, keplerian_L2, keplerian_T2, keplerian_L1, keplerian_T1)

    return keplerian_T1, keplerian_L1, keplerian_T2, keplerian_L2, keplerian_T3, keplerian_L3, keplerian_T4, keplerian_R1, keplerian_T5, keplerian_R2, keplerian_M

keplerian_matrices = run_keplerian()

for i in range(len(keplerian_matrices)-1):
  print(f"Element Matrix {i+1}:")
  print(keplerian_matrices[i])
  print("\n")

print("System Matrix (M):")
print(keplerian_matrices[-1])

Element Matrix 1:
[[1.   0.02]
 [0.   1.  ]]


Element Matrix 2:
[[  1.    0. ]
 [-12.5   1. ]]


Element Matrix 3:
[[1.  0.1]
 [0.  1. ]]


Element Matrix 4:
[[  1.   0.]
 [-50.   1.]]


Element Matrix 5:
[[1.   0.01]
 [0.   1.  ]]


Element Matrix 6:
[[  1.   0.]
 [-40.   1.]]


Element Matrix 7:
[[1.    0.025]
 [0.    1.   ]]


Element Matrix 8:
[[ 1.   0. ]
 [18.   0.1]]


Element Matrix 9:
[[1.     0.0001]
 [0.     1.    ]]


Element Matrix 10:
[[   1.    0.]
 [-180.   10.]]


System Matrix (M):
[[ 0.0001000000000001 -0.100242          ]
 [ 9.982000000000003  -6.156440000000001 ]]


In [ ]:
def run_galilean():
    n_air = 1.0

    # Element 1
    d1 = 0.06

    # Element 2
    f1 = 0.08

    # Element 3
    d2 = 0.05

    # Element 4
    f2 = -0.03

    # Element 5
    d3 = 0.01

    # Element 6
    f = 0.025

    # Element 7
    d4 = 0.025

    # Element 8
    n1 = 10.0
    R1 = -0.05

    # Element 9
    d5 = 0.0001

    # Element 10
    n2 = 1.0
    R2 = -0.05

    galilean_T1 = T_term(d1)
    galilean_L1 = thin_lens_f_term(f1)

    galilean_T2 = T_term(d2)
    galilean_L2 = thin_lens_f_term(f2)

    galilean_T3 = T_term(d3)
    galilean_L3 = thin_lens_f_term(f)

    galilean_T4 = T_term(d4)
    galilean_R1 = R_term_spherical(n_air, n1, R1)

    galilean_T5 = T_term(d5)
    galilean_R2 = R_term_spherical(n1, n2, R2)

    galilean_M = M_sys(galilean_R2, galilean_T5, galilean_R1, galilean_T4, galilean_L3, galilean_T3, galilean_L2, galilean_T2, galilean_L1, galilean_T1)

    return galilean_T1, galilean_L1, galilean_T2, galilean_L2, galilean_T3, galilean_L3, galilean_T4, galilean_R1, galilean_T5, galilean_R2, galilean_M

galilean_matrices = run_galilean()

for i in range(len(galilean_matrices)-1):
  print(f"Element Matrix {i+1}:")
  print(galilean_matrices[i])
  print("\n")

print("System Matrix (M):")
print(galilean_matrices[-1])

Element Matrix 1:
[[1.   0.06]
 [0.   1.  ]]


Element Matrix 2:
[[  1.    0. ]
 [-12.5   1. ]]


Element Matrix 3:
[[1.   0.05]
 [0.   1.  ]]


Element Matrix 4:
[[ 1.                 0.               ]
 [33.333333333333336  1.               ]]


Element Matrix 5:
[[1.   0.01]
 [0.   1.  ]]


Element Matrix 6:
[[  1.   0.]
 [-40.   1.]]


Element Matrix 7:
[[1.    0.025]
 [0.    1.   ]]


Element Matrix 8:
[[ 1.   0. ]
 [18.   0.1]]


Element Matrix 9:
[[1.     0.0001]
 [0.     1.    ]]


Element Matrix 10:
[[   1.    0.]
 [-180.   10.]]


System Matrix (M):
[[ -0.0001500000000001   0.0667736666666666]
 [-14.973000000000003   -1.3192600000000012]]
